# Calling LLM APIs and getting structured output

Machine Learning Engineer track, lesson MLE8. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0. The questions are about the site's own SQL lessons, also CC BY 4.0.

**Goals:** send a request to a language model and read the reply and its token counts, control temperature, require JSON and validate it, keep prompts as versioned code with tests, and measure a cost and latency budget.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. No key and no account are needed: the model is open and runs in this notebook. A free CPU runtime is enough (the model is about 3.3 GB to download, and the cell of Unit 3, which asks it 32 questions and the retries, takes about 8 minutes on two CPU threads, not timed on Colab); a GPU runtime may be faster, but it was not timed.

## Setup

In [ ]:
import importlib.metadata
import json
import math
import numbers
import statistics
import subprocess
import sys
import time
from collections import Counter
from typing import Literal

for pin in ["transformers==5.24.1", "pydantic==2.14.4"]:                      # the versions this notebook was run with
    package, version = pin.split("==")
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:                                                  # install the pinned version when Colab has another
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pin], check=True)

import torch
from pydantic import BaseModel, ValidationError
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 2026       # the seed of the whole track
torch.set_num_threads(2)       # two threads, the size of a free Colab CPU

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## The model and the request function

A small open model that follows instructions: `granite-4.0-1b` (1.63 billion numbers), made by IBM, Apache 2.0, and downloaded without an account. It is pinned to one revision of the model repository. Run this cell as it is: it downloads the model once and loads it on the GPU when there is one, on the CPU otherwise.

`chat` is the one function that sends a request. It takes the messages (a list of roles and texts), the settings, and returns the reply and what a hosted API reports with it: the prompt tokens, the completion tokens (the end marker is not counted) and the seconds. It passes its own sampling settings, so the defaults that ship with the model do not matter. A hosted API takes the same three things: messages, settings, and the counts in its reply.

In [ ]:
LLM_NAME = "ibm-granite/granite-4.0-1b"
LLM_REVISION = "6a7381ba1f54d684ff508d991aeb7dc580157103"                    # the exact version of the model files
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(LLM_NAME, revision=LLM_REVISION)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, revision=LLM_REVISION, dtype=torch.float32).to(DEVICE).eval()
END_MARKS = {tok.eos_token_id}                                              # the end-of-text marker: <|end_of_text|>


def chat(messages, max_new_tokens=100, temperature=0.0, seed=None):
    """One request. temperature 0 picks the most likely token every time; above 0 it samples."""
    prompt = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].to(DEVICE)
    settings = {"do_sample": False, "repetition_penalty": 1.0}
    if temperature > 0:
        settings = {"do_sample": True, "temperature": temperature, "top_p": 1.0, "top_k": 0, "repetition_penalty": 1.0}
        torch.manual_seed(SEED if seed is None else seed)
    start = time.perf_counter()
    with torch.no_grad():
        out = llm.generate(input_ids=prompt, attention_mask=torch.ones_like(prompt), max_new_tokens=max_new_tokens, **settings)
    seconds = time.perf_counter() - start
    written = out[0][prompt.shape[1]:].tolist()
    return {"text": tok.decode(written, skip_special_tokens=True), "prompt_tokens": prompt.shape[1],
            "completion_tokens": sum(1 for t in written if t not in END_MARKS), "seconds": seconds}


print(sum(p.numel() for p in llm.parameters()), DEVICE)

## The questions

16 questions written for this lesson, two for each of eight lessons of the SQL stage, each with the lesson that answers it (the lessons are those of lesson MLE7). `QUESTIONS` and `EXPECTED` are the questions and their lessons, and `LESSONS` the eight names. Run this cell as it is.

In [ ]:
LESSONS = ['null-handling', 'group-by-having', 'inner-join', 'left-join-anti-join', 'window-functions', 'ctes', 'dates-and-time', 'case-expressions']
PROBE = [
    ('Why does WHERE customer_id = NULL never return a row?', 'null-handling'),
    ('Which function replaces a missing value with a default?', 'null-handling'),
    ('What is the difference between WHERE and HAVING?', 'group-by-having'),
    ('How many orders does store 2 have in all?', 'group-by-having'),
    ('What happens to a row that has no partner in an inner join?', 'inner-join'),
    ('If a key repeats on both sides, how many rows does a join return?', 'inner-join'),
    ('How do I find customers who never placed an order?', 'left-join-anti-join'),
    ('Why did my LEFT JOIN lose the unmatched rows after I added a WHERE on the right table?', 'left-join-anti-join'),
    ('How can I add a total beside every row without losing rows?', 'window-functions'),
    ('Can I filter on the result of a window function in WHERE?', 'window-functions'),
    ('What is a CTE?', 'ctes'),
    ('Does a CTE stay in the database after the query ends?', 'ctes'),
    ('How do I get the first day of the month of a timestamp?', 'dates-and-time'),
    ('Why does BETWEEN miss orders placed on the last day of a month?', 'dates-and-time'),
    ('What does CASE return when no WHEN is true and there is no ELSE?', 'case-expressions'),
    ('Which WHEN wins when two tests are both true?', 'case-expressions'),
]
QUESTIONS, EXPECTED = [q for q, _ in PROBE], [lesson for _, lesson in PROBE]
print(len(QUESTIONS), "questions,", len(LESSONS), "lessons")

## Unit 1: A request and its reply

**Predict first.** The user message below has a number of tokens. How many tokens will the request count, and how many will the model write? Is the request more or less than the message you wrote?

In [ ]:
messages = [{"role": "user", "content": "In one sentence, what does COALESCE do in SQL?"}]
reply = chat(messages, max_new_tokens=60)
print(reply["text"])
print(len(tok(messages[0]["content"])["input_ids"]), reply["prompt_tokens"], reply["completion_tokens"])
print(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

## Unit 2: Temperature

**Predict first.** The same request is sent five times with a different seed, once at temperature 0 and once at temperature 1.0. How many different replies will each set of five hold?

In [ ]:
request = [{"role": "user", "content": "Suggest a name for a small cafe that sells coffee and SQL books. Reply with the name only."}]
for temperature in [0.0, 1.0]:
    names = [chat(request, max_new_tokens=12, temperature=temperature, seed=seed)["text"].strip() for seed in range(5)]
    print(temperature, len(set(names)), names)

## Unit 3: Ask for JSON, then check it

**Predict first.** The model must route each of the 16 questions to one of the eight lessons and reply as JSON with a `lesson` and a `reason`. Version v1 of the prompt only says to reply with JSON. Version v2 lists the eight allowed names and shows the shape. How many of the 16 replies will pass the `Route` check for each version, and how many will name the right lesson?

In [ ]:
class Route(BaseModel):
    lesson: Literal[tuple(LESSONS)]                                           # one of the eight names, nothing else
    reason: str


PROMPTS = {
    "v1": "Reply with JSON only, with the keys lesson and reason. The lesson is the name of the SQL lesson that answers the question.",
    "v2": "You route learner questions to lessons. Reply with JSON only, like {\"lesson\": \"ctes\", \"reason\": \"one short sentence\"}. "
          "The value of lesson must be exactly one of: " + ", ".join(LESSONS) + ".",
}


def checked(reply):
    """The reply with the lesson it names, or the type of its error."""
    try:
        return {**reply, "lesson": Route.model_validate_json(reply["text"]).lesson, "error": None}
    except ValidationError as problem:
        return {**reply, "lesson": None, "error": problem.errors()[0]["type"]}


def ask(version, question, temperature=0.0):
    """One routing request with the prompt of a version."""
    return checked(chat([{"role": "system", "content": PROMPTS[version]}, {"role": "user", "content": question}], max_new_tokens=80, temperature=temperature))


def repair(version, question, first):
    """One more try for a reply that failed: the model sees its own reply and the type of the error."""
    messages = [{"role": "system", "content": PROMPTS[version]}, {"role": "user", "content": question}, {"role": "assistant", "content": first["text"]},
                {"role": "user", "content": f"That reply failed the check ({first['error']}). Reply again with the JSON only."}]
    return checked(chat(messages, max_new_tokens=80))


def named(row):
    """The lesson a reply names, read from its raw text (None when the text is not JSON)."""
    try:
        return json.loads(row["text"]).get("lesson")
    except ValueError:
        return None


def failures(version):
    """The error types of a version, counted, and how many of its failed replies still name the right lesson."""
    rows = runs[version]
    return dict(Counter(r["error"] for r in rows if r["error"])), sum(named(r) == e for r, e in zip(rows, EXPECTED) if r["error"])


runs = {version: [ask(version, question) for question in QUESTIONS] for version in ["v1", "v2"]}
for version, rows in runs.items():
    print(version, sum(r["error"] is None for r in rows), "pass,", sum(r["lesson"] == e for r, e in zip(rows, EXPECTED)), "right lesson,", sorted({r["error"] for r in rows if r["error"]}))
for version in runs:
    print(version, *failures(version))
for row in runs["v1"][:2]:
    print(row["error"], row["completion_tokens"], repr(row["text"][:48]))
failed = [i for i, row in enumerate(runs["v2"]) if row["error"]]
fixes = {i: repair("v2", QUESTIONS[i], runs["v2"][i]) for i in failed}                # one more try for each failed reply
for i, second in fixes.items():
    print(runs["v2"][i]["error"], "->", second["error"], second["lesson"] == EXPECTED[i])

## Unit 4: Prompts as code, tests and a budget

**Predict first.** Version v3 changes one phrase of v2: the reason may have at most 8 words. A help box handles 400,000 questions a month and the team allows $90 a month for routing, which is $0.225 for 1,000 requests, at the assumed prices in the cell. Which of v2 and v3 fits the budget, and does v3 still name the right lesson as often as v2?

In [ ]:
PROMPTS["v3"] = PROMPTS["v2"].replace("one short sentence", "at most 8 words")         # the change under test
runs["v3"] = [ask("v3", question) for question in QUESTIONS]

PRICE_IN, PRICE_OUT = 1.00, 5.00                                           # dollars per million tokens: assumed for this lesson, not any provider's price
BUDGET = 90 / 400_000 * 1000                                               # an invented budget (400,000 questions a month, $90 for routing), in dollars for 1,000 requests
for version in ["v2", "v3"]:
    rows = runs[version]
    prompt_mean = statistics.mean(r["prompt_tokens"] for r in rows)
    completion_mean = statistics.mean(r["completion_tokens"] for r in rows)
    cost = (prompt_mean * PRICE_IN + completion_mean * PRICE_OUT) / 1_000_000 * 1000
    print(version, sum(r["error"] is None for r in rows), sum(r["lesson"] == e for r, e in zip(rows, EXPECTED)), round(prompt_mean), round(completion_mean),
          round(cost, 3), cost <= BUDGET, round(statistics.median(r["seconds"] for r in rows), 1))
print("v3", *failures("v3"))

The same request to a hosted API (optional, not needed for any exercise). Keep the key out of the notebook: in Colab, put it in the Secrets panel and read it with `google.colab.userdata`. The address, the headers and the names of the fields come from your provider's documentation; the three lines below are what changes. This cell does nothing until you set `HOSTED_URL`.

In [ ]:
HOSTED_URL = ""                                                            # the address your provider documents for chat requests
HOSTED_HEADERS = {}                                                        # for example the key, read from the Secrets panel, never typed here
HOSTED_BODY = {}                                                           # the model name, the messages and the settings, in your provider's field names
if HOSTED_URL:
    import requests

    answer = requests.post(HOSTED_URL, headers=HOSTED_HEADERS, json=HOSTED_BODY, timeout=60)
    answer.raise_for_status()                                              # a failed call is an error status, so check it before reading the body
    print(answer.json())

## Finish it

### Exercise 1: Finish it

**Task:** **Read the error types.** Five replies are written for this exercise, not by the model. Check each with `Route.model_validate_json` and put the type of its first error in the list `answer_1`, in order, with `None` for a reply that passes. The type is `problem.errors()[0]["type"]` for the `ValidationError` named `problem`.

**Expected:** `answer_1` is a list of five items, each `None` or a short text.

In [ ]:
FENCE = "`" * 3                                                          # three backticks: the marks of a code block
replies = ['{"lesson": "ctes", "reason": "a CTE names a step"}',
           FENCE + 'json\n{"lesson": "ctes", "reason": "a CTE names a step"}\n' + FENCE,
           '{"lesson": "CTE", "reason": "names a step"}',
           '{"lesson": "ctes"}',
           '{"lesson": "ctes", "reason": "names a step']
answer_1 = []
for text in replies:
    try:
        Route.model_validate_json(text)
        answer_1.append(None)
    except ValidationError as problem:
        pass  # add the type of the first error to answer_1

In [ ]:
_ = check("Exercise 1", "answer_1", [None, "json_invalid", "literal_error", "missing", "json_invalid"])

<details>
<summary>Hint</summary>

In the `except` branch, append `problem.errors()[0]["type"]`.

</details>

<details>
<summary>Half-solution</summary>

```python
    except ValidationError as problem:
        answer_1.append(...)
```

</details>

<details>
<summary>Solution</summary>

```python
FENCE = "`" * 3
replies = ['{"lesson": "ctes", "reason": "a CTE names a step"}',
           FENCE + 'json\n{"lesson": "ctes", "reason": "a CTE names a step"}\n' + FENCE,
           '{"lesson": "CTE", "reason": "names a step"}',
           '{"lesson": "ctes"}',
           '{"lesson": "ctes", "reason": "names a step']
answer_1 = []
for text in replies:
    try:
        Route.model_validate_json(text)
        answer_1.append(None)
    except ValidationError as problem:
        answer_1.append(problem.errors()[0]["type"])
```

[None, 'json_invalid', 'literal_error', 'missing', 'json_invalid']: the first reply passes. The fenced one is not JSON. `CTE` is JSON but not one of the eight names. The fourth has no `reason`. The last stops in the middle of a string, which is how a reply looks when it reaches the limit of `max_new_tokens`.

</details>

## On your own

### Exercise 2: On your own

**Task:** **A bill from the rows.** Before you start, say in one sentence what you expect: the prompt tokens of v2 outnumber its completion tokens, and the new prices charge completion tokens eight times as much as prompt tokens. Which side of the bill is bigger? Then write the cell yourself, in the empty cell below: from the 16 requests of `runs["v2"]`, take the mean of the prompt tokens and the mean of the completion tokens, and put the cost of 1,000 requests at $0.25 per million prompt tokens and $2.00 per million completion tokens (assumed prices) in dollars, rounded to 4 decimals, in `answer_2`.

**Expected:** `answer_2` is a number below 0.2.

In [ ]:
_ = check("Exercise 2", "answer_2", 0.0694, tol=5e-5)

<details>
<summary>Hint</summary>

The cost of a request is its prompt tokens times the price of one prompt token plus its completion tokens times the price of one completion token. Prices are per million tokens, and the question asks for 1,000 requests.

</details>

<details>
<summary>Half-solution</summary>

```python
rows = runs["v2"]
prompt_mean = statistics.mean(r["prompt_tokens"] for r in rows)
...
```

</details>

<details>
<summary>Solution</summary>

```python
rows = runs["v2"]
prompt_mean = statistics.mean(r["prompt_tokens"] for r in rows)
completion_mean = statistics.mean(r["completion_tokens"] for r in rows)
answer_2 = round((prompt_mean * 0.25 + completion_mean * 2.00) / 1_000_000 * 1000, 4)
```

0.0694. Version v2 averages about 100 prompt tokens and 22 completion tokens, so at these prices the prompt side costs about 100 x 0.25 = 25 and the completion side 22 x 2.00 = 44, in millionths of a dollar for one request: the smaller count is the bigger part of the bill. For 1,000 requests that is $0.0694.

</details>

## Check yourself

### Exercise 3: Check yourself

**Task:** **Spot the flaw.** A colleague reports the tokens that version v2 used for the 16 questions as the sum of the prompt tokens and completion tokens of the 16 requests in `runs["v2"]`. Change the code so that it counts every request that was sent for those 16 questions, and put the total in `answer_3`.

**Expected:** `answer_3` is a whole number.

In [ ]:
total_tokens = sum(r["prompt_tokens"] + r["completion_tokens"] for r in runs["v2"])        # colleague: "16 questions, 16 requests"
answer_3 = total_tokens
# change the code so that it counts every request that was sent for the 16 questions

In [ ]:
_ = check("Exercise 3", "answer_3", 2647, traps={1949: "That is the first request of each question only: a retry is a request too."})

<details>
<summary>Hint</summary>

Unit 3 made one more request for each reply that failed the check. Their tokens are in `fixes`, a dictionary of replies.

</details>

<details>
<summary>Half-solution</summary>

```python
total_tokens = sum(r["prompt_tokens"] + r["completion_tokens"] for r in runs["v2"])
answer_3 = total_tokens + ...
```

</details>

<details>
<summary>Solution</summary>

```python
total_tokens = sum(r["prompt_tokens"] + r["completion_tokens"] for r in runs["v2"])
total_tokens += sum(r["prompt_tokens"] + r["completion_tokens"] for r in fixes.values())
answer_3 = total_tokens
```

The answer is 2647. The 16 first requests used 1,949 tokens. The five retries of unit 3 are five more requests of 698 tokens in all, and each prompt holds the failed reply and the error as well as the question. A retry costs like any request, and the cost of unit 4 was for the first request of each question only.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")